# Adidas Supply Planning Agent — Standalone Colab Notebook

Fully self-contained version: no `git clone`, no dependency on the GitHub repo being reachable. Every module's source code is embedded directly in this notebook and written to disk when you run the cells.

Uses Google Gemini's free API tier (no OpenAI billing required).

**Note on state:** Colab runtimes are ephemeral — the database and FAISS index reset on restart. Fine for demos/testing.

**Keeping this in sync:** this notebook duplicates the code from [isalgi/buying-planning-agent](https://github.com/isalgi/buying-planning-agent). If that repo changes, this notebook needs to be regenerated/updated separately.

Run the cells below in order.

In [ ]:
# 1. Create a working directory
import os
os.makedirs("/content/buying_planning_agent", exist_ok=True)
os.makedirs("/content/buying_planning_agent/agents", exist_ok=True)
os.makedirs("/content/buying_planning_agent/data", exist_ok=True)
%cd /content/buying_planning_agent

In [ ]:
%%writefile requirements.txt
# Core LangChain ecosystem
langchain>=0.3.0,<0.4.0
langchain-core>=0.3.0,<0.4.0
langchain-community>=0.3.0,<0.4.0
langchain-openai>=0.2.0,<0.3.0
langgraph>=0.2.0,<0.3.0
langsmith>=0.1.100,<0.2.0

# OpenAI
openai>=1.45.0,<2.0.0

# UI
streamlit>=1.38.0,<2.0.0

# Vector stores
faiss-cpu>=1.8.0,<2.0.0
chromadb>=0.5.5,<0.6.0

# Utilities
python-dotenv>=1.0.0
pydantic>=2.9.0,<3.0.0
pydantic-settings>=2.5.0
tiktoken>=0.8.0
numpy>=1.26.0,<2.0.0

# Additional required packages
requests>=2.32.0
typing-extensions>=4.12.0
tenacity>=9.0.0
packaging>=24.0
PyYAML>=6.0
SQLAlchemy>=2.0.0

grandalf
gradio

In [ ]:
# 2. Install dependencies
# pydantic<2.11 is required: newer pydantic breaks gradio 4.44.1's API schema introspection
!pip install -q -r requirements.txt
!pip install -q "pydantic<2.11"

## Get a free Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Click "Create API key" and copy it.
3. Run the next cell **on its own** (not via "Run all") and wait for the input box to appear at the top before pasting. The cell after it does a live test call so you'll know right away if the key works.

In [ ]:
# 3. Configure environment variables
import getpass

while True:
    google_api_key = getpass.getpass("Enter your Google AI Studio API key: ").strip()
    if not google_api_key:
        print("\u274c Empty input \u2014 the key box may not have been ready. Try again.")
        continue
    break

env_content = f"""GOOGLE_API_KEY={google_api_key}
LANGSMITH_API_KEY=your_key_here
LANGCHAIN_PROJECT=adidas-supply-planning
LANGCHAIN_TRACING_V2=false
LANGSMITH_TRACING=False
LANGSMITH_ENDPOINT=https://api.smith.langchain.com/
LANGSMITH_PROJECT=adidas-supply-planning
"""

with open(".env", "w") as f:
    f.write(env_content)

print(f"\u2713 .env saved (key length: {len(google_api_key)}). Run the next cell to verify it actually works.")

(Optional) If you also want LangSmith tracing, get a free key at https://smith.langchain.com/ (Settings → API Keys) and re-run the cell above, replacing `your_key_here` and setting the tracing flags to `true`/`True`.

In [ ]:
# 3b. Sanity-check the key actually works before initializing anything
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
try:
    _resp = _client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": "Say OK"}],
    )
    print("\u2713 Key works:", _resp.choices[0].message.content)
except Exception as e:
    print("\u274c Key test failed:", e)
    print("Re-run the previous cell and paste the key again.")

## Write the application source files

Each cell below recreates one module from the repo, exactly as-is.

In [ ]:
%%writefile config.py
"""Configuration module for environment variables and settings."""
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
load_dotenv()

# Gemini Configuration (used via Google's OpenAI-compatible endpoint, free tier)
OPENAI_API_KEY = os.getenv("GOOGLE_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# LangSmith Configuration
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "adidas-supply-planning")
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"

# Database Configuration
DB_PATH = Path("data/adidas_supply.db")
DB_PATH.parent.mkdir(exist_ok=True)

# RAG Configuration
RAG_DOCUMENTS_PATH = Path("data/supply_docs")
RAG_DOCUMENTS_PATH.mkdir(exist_ok=True)
EMBEDDING_MODEL = "gemini-embedding-001"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RESULTS = 3

# Model Configuration
LLM_MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.7

if __name__ == "__main__":
    # Test configuration
    print("✓ Configuration loaded successfully")
    print(f"  LangSmith Project: {LANGSMITH_PROJECT}")
    print(f"  Database Path: {DB_PATH}")
    print(f"  RAG Documents Path: {RAG_DOCUMENTS_PATH}")
    print(f"  LLM Model: {LLM_MODEL}")

In [ ]:
%%writefile db.py
"""Database module for persistent storage with multi-user support."""
import sqlite3
import json
from datetime import datetime
from contextlib import contextmanager
from typing import List, Dict, Optional
import threading
from config import DB_PATH

# Thread-local storage for database connections
_thread_local = threading.local()

class DatabaseManager:
    """Thread-safe database manager for SQLite operations."""
    
    def __init__(self, db_path: str = DB_PATH):
        self.db_path = str(db_path)
        self._init_db()
    
    def _get_connection(self):
        """Get thread-local database connection."""
        if not hasattr(_thread_local, 'connection'):
            _thread_local.connection = sqlite3.connect(
                self.db_path,
                timeout=30,  # Wait up to 30s for lock
                check_same_thread=False  # We're managing threads manually
            )
            _thread_local.connection.row_factory = sqlite3.Row
        return _thread_local.connection
    
    @contextmanager
    def get_cursor(self):
        """Context manager for database cursors with automatic commit/rollback."""
        conn = self._get_connection()
        cursor = conn.cursor()
        try:
            yield cursor
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cursor.close()
    
    def _init_db(self):
        """Initialize database tables if they don't exist."""
        with self.get_cursor() as cursor:
            # Create conversations table
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    user_query TEXT NOT NULL,
                    assistant_response TEXT NOT NULL,
                    agent_used TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    metadata TEXT
                )
            """)
            
            # Create index for faster session queries
            cursor.execute("""
                CREATE INDEX IF NOT EXISTS idx_session_timestamp 
                ON conversations(session_id, timestamp)
            """)
    
    def save_conversation(self, session_id: str, user_query: str, 
                         assistant_response: str, agent_used: str = None,
                         metadata: Dict = None):
        """Save a conversation turn to database."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                INSERT INTO conversations 
                (session_id, user_query, assistant_response, agent_used, metadata)
                VALUES (?, ?, ?, ?, ?)
            """, (
                session_id, 
                user_query, 
                assistant_response, 
                agent_used,
                json.dumps(metadata) if metadata else None
            ))
    
    def load_session_history(self, session_id: str, limit: int = 50) -> List[Dict]:
        """Load conversation history for a session."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                SELECT user_query, assistant_response, agent_used, timestamp, metadata
                FROM conversations
                WHERE session_id = ?
                ORDER BY timestamp DESC
                LIMIT ?
            """, (session_id, limit))
            
            rows = cursor.fetchall()
            return [
                {
                    'user_query': row['user_query'],
                    'assistant_response': row['assistant_response'],
                    'agent_used': row['agent_used'],
                    'timestamp': row['timestamp'],
                    'metadata': json.loads(row['metadata']) if row['metadata'] else {}
                }
                for row in rows
            ]
    
    def get_all_sessions(self) -> List[str]:
        """Get all unique session IDs."""
        with self.get_cursor() as cursor:
            cursor.execute("SELECT DISTINCT session_id FROM conversations")
            return [row['session_id'] for row in cursor.fetchall()]
    
    def delete_session(self, session_id: str):
        """Delete a session and all its conversations."""
        with self.get_cursor() as cursor:
            cursor.execute("DELETE FROM conversations WHERE session_id = ?", (session_id,))


# Global database instance
db_manager = DatabaseManager()

if __name__ == "__main__":
    # Test database functionality
    print("Testing Database Module...")
    
    # Test session
    test_session = "test_session_123"
    
    # Save test conversation
    db_manager.save_conversation(
        session_id=test_session,
        user_query="What is the demand forecast for running shoes?",
        assistant_response="Based on historical data, demand is expected to increase by 15%.",
        agent_used="demand_forecast",
        metadata={"test": True}
    )
    
    # Load history
    history = db_manager.load_session_history(test_session)
    print(f"✓ Saved and loaded {len(history)} conversations")
    
    # List all sessions
    sessions = db_manager.get_all_sessions()
    print(f"✓ Active sessions: {sessions}")
    
    # Cleanup
    db_manager.delete_session(test_session)
    print("✓ Test session cleaned up")

In [ ]:
%%writefile rag.py
"""RAG module for document retrieval and context injection."""
import os
import pickle
from typing import List, Dict, Any
from pathlib import Path
import numpy as np
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langsmith import traceable
from config import (
    OPENAI_API_KEY,
    GEMINI_BASE_URL,
    EMBEDDING_MODEL,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    TOP_K_RESULTS,
    RAG_DOCUMENTS_PATH
)

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=GEMINI_BASE_URL,
    check_embedding_ctx_length=False,
)

class RAGSystem:
    """Lightweight RAG system for Adidas supply planning documents."""
    
    def __init__(self, persist_directory: str = "data/faiss_index"):
        self.persist_directory = persist_directory
        self.vector_store = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
        )
        
        # Load or create vector store
        self._initialize_vector_store()
    
    def _initialize_vector_store(self):
        """Initialize FAISS vector store from existing index or create new."""
        if os.path.exists(self.persist_directory):
            try:
                self.vector_store = FAISS.load_local(
                    self.persist_directory, 
                    embeddings,
                    allow_dangerous_deserialization=True
                )
                print(f"✓ Loaded existing FAISS index from {self.persist_directory}")
            except Exception as e:
                print(f"! Could not load existing index: {e}")
                self.vector_store = None
        
        if self.vector_store is None:
            # Create sample documents if none exist
            self._create_sample_documents()
    
    def _create_sample_documents(self):
        """Create sample Adidas supply planning documents."""
        documents = [
            Document(
                page_content="""Adidas Demand Forecasting Guidelines:
                - Use historical sales data from last 3 years
                - Account for seasonal trends: Q4 has 30% higher demand
                - Regional variations: Europe 40% of sales, North America 35%, Asia 25%
                - New product launches increase baseline demand by 15-20%
                - Promotional periods can spike demand by 50% temporarily""",
                metadata={"category": "demand_forecast", "source": "guidelines_v2"}
            ),
            Document(
                page_content="""Size Curve Optimization Best Practices:
                - Running shoes: sizes 8-10 represent 60% of sales
                - Lifestyle shoes: broader distribution, sizes 7-11 represent 70%
                - Regional differences: Asian markets need smaller sizes (shift -1.5 sizes)
                - Review size curves monthly based on sell-through rates
                - Safety stock for popular sizes should be 20% higher""",
                metadata={"category": "size_curve", "source": "optimization_guide"}
            ),
            Document(
                page_content="""Price Optimization Strategy:
                - Premium products (Ultraboost): 30% margin target
                - Core products (Superstar): 45% margin target
                - Discount thresholds: max 30% for seasonal items
                - Dynamic pricing based on competitor analysis
                - Bundle pricing: 15% discount for 2+ items""",
                metadata={"category": "price_optimization", "source": "pricing_strategy"}
            ),
            Document(
                page_content="""Inventory Management Rules:
                - Safety stock: 15% of forecasted demand
                - Reorder point: 30 days of inventory
                - Seasonal build-up: start 60 days before season
                - Clearance: 40% discount after 120 days in stock
                - Cross-docking for high-volume items""",
                metadata={"category": "inventory", "source": "inventory_policy"}
            ),
            Document(
                page_content="""Supply Chain Constraints:
                - Production lead time: 45 days from Asia
                - Air freight: 7 days (3x cost)
                - Port capacity: 20% lower during Chinese New Year
                - Raw material availability: 95% typically
                - Factory utilization target: 85%""",
                metadata={"category": "supply_chain", "source": "operations"}
            )
        ]
        
        # Create vector store
        self.add_documents(documents)
        print("✓ Created sample documents and FAISS index")
    
    @traceable(name="rag_add_documents", run_type="chain")
    def add_documents(self, documents: List[Document]):
        """Add documents to the vector store."""
        # Split documents into chunks
        chunks = self.text_splitter.split_documents(documents)
        
        # Create or update vector store
        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(chunks, embeddings)
        else:
            self.vector_store.add_documents(chunks)
        
        # Save to disk
        os.makedirs(os.path.dirname(self.persist_directory), exist_ok=True)
        self.vector_store.save_local(self.persist_directory)
    
    @traceable(name="rag_retrieve", run_type="retriever")
    def retrieve_context(self, query: str, k: int = TOP_K_RESULTS) -> List[Document]:
        """Retrieve relevant documents for a query."""
        if self.vector_store is None:
            return []
        
        docs = self.vector_store.similarity_search(query, k=k)
        return docs
    
    @traceable(name="rag_get_context", run_type="chain")
    def get_context_string(self, query: str) -> str:
        """Get context as a formatted string for prompt injection."""
        docs = self.retrieve_context(query)
        
        if not docs:
            return "No relevant documents found."
        
        context_parts = []
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get('source', 'Unknown')
            category = doc.metadata.get('category', 'General')
            context_parts.append(f"[Document {i} - {category} ({source})]:\n{doc.page_content}\n")
        
        return "\n".join(context_parts)


# Global RAG instance
rag_system = RAGSystem()

if __name__ == "__main__":
    # Test RAG functionality
    print("Testing RAG Module...")
    
    # Test queries
    test_queries = [
        "What are the demand forecast guidelines for running shoes?",
        "How should I optimize size curves for different regions?",
        "What is the pricing strategy for premium products?"
    ]
    
    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        
        # Get context
        context = rag_system.get_context_string(query)
        print(f"Retrieved Context:\n{context}")
        
        # Show retrieval details
        docs = rag_system.retrieve_context(query)
        print(f"\n✓ Retrieved {len(docs)} relevant documents")

In [ ]:
%%writefile router.py
"""Router module for classifying user intent in supply planning queries."""
import os
import json
from typing import Dict, Any
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="router_classification", run_type="chain")
def classify_intent(query: str) -> Dict[str, Any]:
    """
    Classify user query into one of the supply planning categories.
    
    Args:
        query: User's question
    
    Returns:
        Dictionary with classification result and confidence
    """
    system_prompt = """You are an intent classifier for Adidas Supply Planning System.
    Classify the user's query into one of these categories:
    
    1. demand_forecast - Questions about predicting future product demand, sales forecasts, inventory planning
    2. size_curve - Questions about size distribution, size optimization, regional size preferences
    3. price_optimization - Questions about pricing strategy, discounts, margins, promotions
    4. general - General questions not specific to the above categories
    
    Respond with JSON format: {"category": "category_name", "confidence": 0.0-1.0, "reasoning": "brief explanation"}
    """
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.3,  # Lower temperature for classification
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    )
    
    try:
        result = json.loads(response.choices[0].message.content) # type: ignore
    except:
        result = {
            "category": "general",
            "confidence": 0.5,
            "reasoning": "Failed to parse response"
        }
    
    # Add usage info
    result["usage"] = response.usage.model_dump() if response.usage else None
    
    return result

@traceable(name="router_decision", run_type="chain")
def route_query(query: str) -> str:
    """
    Route query to appropriate agent based on intent.
    
    Args:
        query: User's question
    
    Returns:
        Agent name to route to
    """
    classification = classify_intent(query)
    return classification.get("category", "general")

if __name__ == "__main__":
    # Test router functionality
    print("Testing Router Module...")
    
    test_queries = [
        "What will be the demand for running shoes next quarter?",
        "How should I allocate sizes for the European market?",
        "What price should I set for the new collection?",
        "Tell me about Adidas history"
    ]
    
    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        
        # Get classification
        classification = classify_intent(query)
        print(f"Classification: {classification}")
        
        # Get route
        route = route_query(query)
        print(f"Routed to: {route}")

In [ ]:
%%writefile agents/__init__.py


In [ ]:
%%writefile agents/demand_agent.py
"""Demand Forecasting Agent for Adidas supply planning."""
import os
from typing import Dict, Any
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="demand_forecast_agent", run_type="chain")
def demand_forecast_agent(query: str, context: str = None) -> Dict[str, Any]:
    """
    Demand Forecasting Agent - Analyzes and predicts product demand.
    
    Args:
        query: User query about demand forecasting
        context: Retrieved RAG context
    
    Returns:
        Dictionary with response and metadata
    """
    system_prompt = """You are Adidas's Demand Forecasting Expert. Your role is to:
    - Analyze historical sales data and market trends
    - Provide accurate demand predictions for products
    - Consider seasonal patterns, regional variations, and promotions
    - Recommend inventory levels based on forecasts
    - Highlight risks and opportunities in demand planning
    
    Use the provided context documents to inform your responses.
    Be specific with numbers and percentages when possible."""
    
    # Build the prompt with context
    user_prompt = f"""Context from Adidas documents:
    {context if context else 'No specific context provided.'}
    
    User Question: {query}
    
    Please provide a detailed demand forecast analysis."""
    
    # Call OpenAI with tracing
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    return {
        "agent": "demand_forecast",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    # Test the agent independently
    print("Testing Demand Forecasting Agent...")
    
    test_query = "What should be the demand forecast for Ultraboost running shoes in Q4?"
    test_context = """Adidas Demand Forecasting Guidelines:
    - Q4 has 30% higher demand due to holiday season
    - Running shoes typically see 15% growth year-over-year
    - Ultraboost is a premium product with 25% market share in performance running"""
    
    result = demand_forecast_agent(test_query, test_context)
    
    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")

In [ ]:
%%writefile agents/price_agent.py
"""Price Optimization Agent for Adidas supply planning."""
import os
from typing import Dict, Any
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="price_optimization_agent", run_type="chain")
def price_optimization_agent(query: str, context: str = None) -> Dict[str, Any]:
    """
    Price Optimization Agent - Optimizes pricing strategy for products.
    
    Args:
        query: User query about price optimization
        context: Retrieved RAG context
    
    Returns:
        Dictionary with response and metadata
    """
    system_prompt = """You are Adidas's Price Optimization Expert. Your role is to:
    - Determine optimal pricing strategies for different product categories
    - Balance margin targets with market competitiveness
    - Recommend promotional discounts and timing
    - Analyze price elasticity and demand sensitivity
    - Provide bundle pricing recommendations
    
    Use the provided context documents to inform your pricing recommendations.
    Include specific price points and margins when relevant."""
    
    # Build the prompt with context
    user_prompt = f"""Context from Adidas documents:
    {context if context else 'No specific context provided.'}
    
    User Question: {query}
    
    Please provide detailed pricing recommendations."""
    
    # Call OpenAI with tracing
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    return {
        "agent": "price_optimization",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    # Test the agent independently
    print("Testing Price Optimization Agent...")
    
    test_query = "What should be the pricing strategy for the new NMD collection?"
    test_context = """Adidas Pricing Strategy:
    - Premium lifestyle: 30% margin target for new releases
    - Limited editions can command 20% premium
    - Bundle 2+ items for 15% discount
    - Monitor competitor pricing weekly
    - Seasonal items: full price first 60 days, then 20% discount"""
    
    result = price_optimization_agent(test_query, test_context)
    
    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")

In [ ]:
%%writefile agents/size_curve_agent.py
"""Size Curve Optimization Agent for Adidas supply planning."""
import os
from typing import Dict, Any
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="size_curve_agent", run_type="chain")
def size_curve_agent(query: str, context: str = None) -> Dict[str, Any]:
    """
    Size Curve Optimization Agent - Optimizes size distribution for products.
    
    Args:
        query: User query about size curve optimization
        context: Retrieved RAG context
    
    Returns:
        Dictionary with response and metadata
    """
    system_prompt = """You are Adidas's Size Curve Optimization Specialist. Your role is to:
    - Analyze optimal size distributions for different product categories
    - Account for regional and demographic variations
    - Recommend size curves based on historical sales data
    - Minimize stockouts and overstock across sizes
    - Adjust curves for specific product types (running, lifestyle, training)
    
    Use the provided context documents to inform your recommendations.
    Provide specific percentages for size distributions."""
    
    # Build the prompt with context
    user_prompt = f"""Context from Adidas documents:
    {context if context else 'No specific context provided.'}
    
    User Question: {query}
    
    Please provide detailed size curve recommendations."""
    
    # Call OpenAI with tracing
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    return {
        "agent": "size_curve",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    # Test the agent independently
    print("Testing Size Curve Optimization Agent...")
    
    test_query = "What size curve should I use for Superstar shoes in the Asian market?"
    test_context = """Adidas Size Curve Best Practices:
    - Lifestyle shoes like Superstar have broader size distribution
    - Asian markets need sizes 1.5 smaller on average
    - Popular sizes 7-9 represent 70% of sales in Asia
    - Unisex styles need adjusted curves for gender differences"""
    
    result = size_curve_agent(test_query, test_context)
    
    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")

In [ ]:
%%writefile graph.py
"""LangGraph workflow for Adidas Supply Planning System."""
from typing import Dict, Any, Literal
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict
from langsmith import traceable
import json

# Import agents
from agents.demand_agent import demand_forecast_agent
from agents.size_curve_agent import size_curve_agent
from agents.price_agent import price_optimization_agent
from router import route_query, classify_intent
from rag import rag_system
from db import db_manager
from openai import OpenAI
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL

client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

# Define state schema with conversation context
class AgentState(TypedDict):
    """State for the LangGraph agent workflow."""
    query: str
    session_id: str
    conversation_context: str
    intent: str
    intent_confidence: float
    intent_reasoning: str
    rag_context: str
    agent_response: str
    agent_used: str
    usage: Dict
    error: str

@traceable(name="inject_context", run_type="chain")
def inject_context_node(state: AgentState) -> AgentState:
    """Node to inject RAG context based on intent."""
    try:
        # Get relevant context - include conversation context if available
        query = state["query"]
        context = state.get("conversation_context", "")
        
        # Enhance RAG with conversation context if available
        enhanced_query = query
        if context:
            # Use the last user message from context to enhance the query
            context_lines = context.split('\n')
            if context_lines:
                last_user_msg = None
                for line in reversed(context_lines):
                    if line.startswith("User:"):
                        last_user_msg = line.replace("User:", "").strip()
                        break
                if last_user_msg:
                    enhanced_query = f"Previous question: {last_user_msg}\nCurrent question: {query}"
        
        # Get relevant context
        context = rag_system.get_context_string(enhanced_query)
        state["rag_context"] = context
    except Exception as e:
        state["rag_context"] = ""
        state["error"] = f"RAG error: {str(e)}"
    
    return state

@traceable(name="router_node", run_type="chain")
def router_node(state: AgentState) -> AgentState:
    """Node to classify intent and route to appropriate agent with conversation context."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")
        
        # Enhance intent classification with conversation context
        enhanced_query = query
        if context:
            # Add context to help with intent classification
            context_lines = context.split('\n')
            recent_exchanges = context_lines[-4:] if len(context_lines) > 4 else context_lines
            enhanced_query = f"""Previous conversation:
{chr(10).join(recent_exchanges)}

Current question: {query}"""
        
        classification = classify_intent(enhanced_query)
        state["intent"] = classification.get("category", "general")
        state["intent_confidence"] = classification.get("confidence", 0.0)
        state["intent_reasoning"] = classification.get("reasoning", "")
        state["usage"] = classification.get("usage")
    except Exception as e:
        state["intent"] = "general"
        state["error"] = f"Router error: {str(e)}"
    
    return state

@traceable(name="demand_agent_node", run_type="chain")
def demand_agent_node(state: AgentState) -> AgentState:
    """Node for demand forecasting agent with conversation context."""
    query = state["query"]
    rag_context = state["rag_context"]
    conversation_context = state.get("conversation_context", "")
    
    # Pass both RAG context and conversation context to agent
    enhanced_context = rag_context
    if conversation_context:
        enhanced_context = f"""Previous conversation context:
{conversation_context}

Relevant documentation:
{rag_context}"""
    
    result = demand_forecast_agent(query, enhanced_context)
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="size_curve_agent_node", run_type="chain")
def size_curve_agent_node(state: AgentState) -> AgentState:
    """Node for size curve optimization agent with conversation context."""
    query = state["query"]
    rag_context = state["rag_context"]
    conversation_context = state.get("conversation_context", "")
    
    enhanced_context = rag_context
    if conversation_context:
        enhanced_context = f"""Previous conversation context:
{conversation_context}

Relevant documentation:
{rag_context}"""
    
    result = size_curve_agent(query, enhanced_context)
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="price_agent_node", run_type="chain")
def price_agent_node(state: AgentState) -> AgentState:
    """Node for price optimization agent with conversation context."""
    query = state["query"]
    rag_context = state["rag_context"]
    conversation_context = state.get("conversation_context", "")
    
    enhanced_context = rag_context
    if conversation_context:
        enhanced_context = f"""Previous conversation context:
{conversation_context}

Relevant documentation:
{rag_context}"""
    
    result = price_optimization_agent(query, enhanced_context)
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="general_agent_node", run_type="chain")
def general_agent_node(state: AgentState) -> AgentState:
    """Node for handling general queries with conversation context."""

    
    query = state["query"]
    context = state.get("conversation_context", "")
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant for Adidas supply planning. Provide general information and guide users to specific agents for detailed queries."}
    ]
    
    # Add conversation context if available
    if context:
        messages.append({"role": "system", "content": f"Previous conversation:\n{context}"})
    
    messages.append({"role": "user", "content": query})
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages
    )
    
    state["agent_response"] = response.choices[0].message.content
    state["agent_used"] = "general"
    state["usage"] = response.usage.model_dump() if response.usage else None
    return state

@traceable(name="save_to_db", run_type="chain")
def save_to_db_node(state: AgentState) -> AgentState:
    """Node to save conversation to database."""
    try:
        db_manager.save_conversation(
            session_id=state["session_id"],
            user_query=state["query"],
            assistant_response=state["agent_response"],
            agent_used=state["agent_used"],
            metadata={
                "intent": state["intent"],
                "confidence": state["intent_confidence"],
                "conversation_context": state.get("conversation_context", ""),  # Save context
                "usage": state["usage"]
            }
        )
    except Exception as e:
        state["error"] = f"Database error: {str(e)}"
    
    return state

def should_continue(state: AgentState) -> Literal["demand", "size_curve", "price", "general", END]:
    """Conditional edge to route to appropriate agent."""
    if state.get("error"):
        return END
    
    intent = state.get("intent", "general")
    
    if intent == "demand_forecast":
        return "demand"
    elif intent == "size_curve":
        return "size_curve"
    elif intent == "price_optimization":
        return "price"
    else:
        return "general"

# Build the graph
def build_supply_planning_graph():
    """Build and compile the LangGraph workflow."""
    
    # Initialize graph
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("router", router_node)
    workflow.add_node("inject_context", inject_context_node)
    workflow.add_node("demand", demand_agent_node)
    workflow.add_node("size_curve", size_curve_agent_node)
    workflow.add_node("price", price_agent_node)
    workflow.add_node("general", general_agent_node)
    workflow.add_node("save_db", save_to_db_node)
    
    # Add edges
    workflow.set_entry_point("router")
    workflow.add_edge("router", "inject_context")
    
    # Conditional routing based on intent
    workflow.add_conditional_edges(
        "inject_context",
        should_continue,
        {
            "demand": "demand",
            "size_curve": "size_curve",
            "price": "price",
            "general": "general"
        }
    )
    
    # Connect agents to database save
    workflow.add_edge("demand", "save_db")
    workflow.add_edge("size_curve", "save_db")
    workflow.add_edge("price", "save_db")
    workflow.add_edge("general", "save_db")
    workflow.add_edge("save_db", END)
    
    # Compile
    return workflow.compile()

# Create global graph instance
supply_planning_graph = build_supply_planning_graph()

if __name__ == "__main__":
    
    ascii_data = supply_planning_graph.get_graph().draw_ascii()
    print(ascii_data)

    # Test queries with context
    test_queries = [
        ("test_session_1", "What is the demand forecast for Ultraboost?"),
        ("test_session_1", "What about for running shoes?"),  # This should now have context
        ("test_session_2", "What size curve should I use for running shoes in Asia?"),
        ("test_session_3", "How should I price the new collection?")
    ]
    
    for session_id, query in test_queries:
        print(f"\nProcessing: {query}")
        print("-" * 50)
        
        # Initialize state with empty context first time
        initial_state = {
            "query": query,
            "session_id": session_id,
            "conversation_context": "",  # In real usage, this would come from UI
            "intent": "",
            "intent_confidence": 0.0,
            "intent_reasoning": "",
            "rag_context": "",
            "agent_response": "",
            "agent_used": "",
            "usage": {},
            "error": ""
        }
        
        # Run graph
        result = supply_planning_graph.invoke(initial_state)
        
        print(f"Intent: {result['intent']} (confidence: {result['intent_confidence']:.2f})")
        print(f"Agent Used: {result['agent_used']}")
        print(f"Response: {result['agent_response'][:100]}...")
        print(f"✓ Graph execution complete")

In [ ]:
%%writefile gradio_ui.py
"""
Adidas Supply Planning System - UI
Run with: python gradio_ui.py
"""
import uuid
import gradio as gr
from graph import supply_planning_graph
from db import db_manager

# Store session per user
sessions = {}

def process_query(query, history, session_id):
    """Process a single query and return response."""
    
    # Create or get session
    if not session_id:
        session_id = str(uuid.uuid4())
        sessions[session_id] = []
    
    # Convert Gradio history to conversation context string
    conversation_context = ""
    if history:
        context_parts = []
        for msg in history:
            # Handle MessageDict format (role, content)
            if isinstance(msg, dict) and "role" in msg and "content" in msg:
                role = "User" if msg["role"] == "user" else "Assistant"
                context_parts.append(f"{role}: {msg['content']}")
        
        # Use last 6 messages (3 exchanges) for context
        conversation_context = "\n".join(context_parts[-6:])
    
    # Initialize state with conversation context
    state = {
        "query": query,
        "session_id": session_id,
        "conversation_context": conversation_context,
        "intent": "",
        "intent_confidence": 0.0,
        "intent_reasoning": "",
        "rag_context": "",
        "agent_response": "",
        "agent_used": "",
        "usage": {},
        "error": ""
    }
    
    try:
        # Run graph with full context
        result = supply_planning_graph.invoke(state)
        
        if result.get("error"):
            response = f"❌ Error: {result['error']}"
        else:
            # Format response
            agent = result['agent_used']
            intent = result['intent']
            confidence = result['intent_confidence']
            answer = result['agent_response']
            
            response = f"""**Agent:** {agent}  
**Intent:** {intent} ({confidence:.2f})  

{answer}"""
            
            if result.get("usage"):
                tokens = result['usage'].get('total_tokens', 0)
                response += f"\n\n---\n*Tokens: {tokens}*"
        
        # Update history with MessageDict format for Gradio 6.8.0
        if history is None:
            history = []
        
        # Use MessageDict format with role and content
        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": response})
        
        # Store in sessions dict
        if session_id in sessions:
            sessions[session_id] = history
        
        return "", history, session_id
        
    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        if history is None:
            history = []
        
        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": error_msg})
        
        if session_id in sessions:
            sessions[session_id] = history
            
        return "", history, session_id

def clear_chat(session_id):
    """Clear chat history for a session."""
    if session_id in sessions:
        sessions[session_id] = []
    return [], session_id

def view_history(session_id):
    """View full session history from DB."""
    if not session_id:
        return "No active session"
    
    history = db_manager.load_session_history(session_id)
    if not history:
        return "No history found"
    
    output = f"## Session History: {session_id}\n\n"
    for msg in history:
        output += f"**You:** {msg['user_query']}\n\n"
        output += f"**AI ({msg['agent_used']}):** {msg['assistant_response']}\n\n"
        output += "---\n\n"
    
    return output

def create_session():
    """Create a new session and return session ID."""
    session_id = str(uuid.uuid4())
    sessions[session_id] = []
    return session_id

# Create Gradio interface
with gr.Blocks(title="Adidas Supply Planning", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 👟 Adidas Supply Planning System
    Ask about demand forecasting, size curves, or pricing optimization.
    """)
    
    # Store session state - initialize with new session
    session_state = gr.State(create_session)
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=500,
                type="messages"
            )
            msg = gr.Textbox(label="Your Question", placeholder="e.g., What is demand forecast for Ultraboost?")
            
            with gr.Row():
                submit = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear Chat")
        
        with gr.Column(scale=1):
            gr.Markdown("### Session Info")
            session_id_display = gr.Textbox(
                label="Session ID", 
                value="",  # Will be updated
                interactive=False
            )
            
            new_session_btn = gr.Button("🆕 New Session", variant="secondary")
            
            gr.Markdown("### Sample Queries")
            sample_queries = gr.Dataset(
                components=[msg],
                samples=[
                    ["What is the demand forecast for Ultraboost in Q4?"],
                    ["Optimize size curve for running shoes in Asia"],
                    ["What about for running shoes in Europe?"],
                    ["What price should I set for the new NMD collection?"],
                    ["Tell me about Adidas supply chain operations"]
                ],
                label="Click to try"
            )
            
            history_btn = gr.Button("View Full History")
            history_output = gr.Markdown()
    
    # Event handlers
    def respond(message, chat_history, session_id):
        if not message:
            return "", chat_history, session_id
        
        # Ensure chat_history is a list
        if chat_history is None:
            chat_history = []
        
        return process_query(message, chat_history, session_id)
    
    def update_session_id(session_id):
        return session_id if session_id else "No active session"
    
    def new_session():
        """Create new session and clear chat."""
        new_id = str(uuid.uuid4())
        sessions[new_id] = []
        return [], new_id, new_id
    
    def clear_chat_handler(session_id):
        """Clear chat history handler."""
        if session_id in sessions:
            sessions[session_id] = []
        return [], session_id
    
    # Connect events
    submit.click(respond, [msg, chatbot, session_state], [msg, chatbot, session_state])
    msg.submit(respond, [msg, chatbot, session_state], [msg, chatbot, session_state])
    
    sample_queries.click(lambda x: x[0], [sample_queries], [msg])
    
    clear.click(clear_chat_handler, [session_state], [chatbot, msg])
    
    new_session_btn.click(new_session, None, [chatbot, session_state, session_id_display])
    
    session_state.change(update_session_id, [session_state], [session_id_display])
    
    history_btn.click(view_history, [session_state], [history_output])
    
    # Initialize session ID display on load
    demo.load(lambda s: s, [session_state], [session_id_display])

if __name__ == "__main__":
    demo.launch(
        share=False,
        server_name="127.0.0.1",
        server_port=7860
    )

In [ ]:
# 4. Initialize the database and RAG (FAISS) index
!python db.py
!python rag.py

In [ ]:
# 5. Launch the Gradio app
# share=True is required on Colab since 127.0.0.1 isn't reachable from your browser —
# Gradio will print a public *.gradio.live link instead.
from gradio_ui import demo

demo.launch(share=True)

## Notes

- **Rate limits:** Gemini's free tier caps `gemini-2.5-flash` at 5 requests/minute. Space out test queries by a few seconds.
- **Restarting:** If you restart the Colab runtime, re-run all cells from the top — the working directory, `.env`, database, and FAISS index are all wiped.
- **Stopping the app:** Use Runtime → Interrupt execution to stop the Gradio server.
- **Updating this notebook:** since the source is embedded rather than cloned, any future fix to the repo needs to be re-applied here too.